In [56]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("PySparkExercises").getOrCreate()

In [57]:
from google.colab import files
uploaded=files.upload

In [58]:
# PART 1
customers_df=spark.read.option("header","true").option("inferSchema","true").csv("customers.csv")

In [59]:
usage_df=spark.read.option("header","true").option("inferSchema","true").csv("usage.csv")

In [60]:
payments_df=spark.read.option("header","true").option("inferSchema","true").csv("payments.csv")

In [61]:
plans_df=spark.read.option("multiline","true").json("plans.json")

In [62]:
customers_df.show()
customers_df.printSchema()

+-----------+-------------+---------+-----------+---+------+-------+--------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|
+-----------+-------------+---------+-----------+---+------+-------+--------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|  Active|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|  Active|
|        110|  Nisha Reddy|    Delhi|      Delhi| 41|Female|   P

In [63]:
usage_df.show()
usage_df.printSchema()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1001|        101|2026-01-01 00:00:00|          45|         900|      120|
|    1002|        102|2026-01-01 00:00:00|          30|         600|       80|
|    1003|        103|2026-01-01 00:00:00|          12|         250|       40|
|    1004|        104|2026-01-01 00:00:00|          55|        1100|      150|
|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|
|    1006|        106|2026-01-01 00:00:00|          28|         500|       60|
|    1007|        107|2026-01-01 00:00:00|          10|         200|       20|
|    1008|        108|2026-01-01 00:00:00|          80|        1600|      250|
|    1009|        109|2026-01-01 00:00:00|          48|         950|      100|
|    1010|        110|2026-01-01 00:00:00|          

In [64]:
payments_df.show()
payments_df.printSchema()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|
|      5008|        108|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5009|        109|2026-01-01 00:00:00|        499|         

In [65]:
plans_df.show()
plans_df.printSchema()

+-------------+--------------------+-----------+-------+------------+
|data_limit_gb|            features|monthly_fee|plan_id|   plan_name|
+-------------+--------------------+-----------+-------+------------+
|           50|{false, National,...|        499|   P101| Smart Basic|
|           75|{true, National, ...|        799|   P102|  Smart Plus|
|           25|{false, NULL, false}|        299|   P103|Budget Saver|
|          100|{true, Internatio...|       1199|   P104| Premium Max|
+-------------+--------------------+-----------+-------+------------+

root
 |-- data_limit_gb: long (nullable = true)
 |-- features: struct (nullable = true)
 |    |-- ott_included: boolean (nullable = true)
 |    |-- roaming: string (nullable = true)
 |    |-- unlimited_calls: boolean (nullable = true)
 |-- monthly_fee: long (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- plan_name: string (nullable = true)



In [66]:
print(customers_df.count())
print(usage_df.count())
print(payments_df.count())
print(plans_df.count())

12
15
15
4


In [67]:
customers_df.write.mode("overwrite").parquet("bronze/customers")
usage_df.write.mode("overwrite").parquet("bronze/usage")
payments_df.write.mode("overwrite").parquet("bronze/payments")
plans_df.write.mode("overwrite").parquet("bronze/plans")

In [68]:
# PART 2
from pyspark.sql.functions import *
customers_df.filter(col("plan_id").isNull()).show()

+-----------+-------------+---------+---------+---+------+-------+------+
|customer_id|customer_name|     city|    state|age|gender|plan_id|status|
+-----------+-------------+---------+---------+---+------+-------+------+
|        112|  Ayesha Khan|Hyderabad|Telangana| 28|Female|   NULL|Active|
+-----------+-------------+---------+---------+---+------+-------+------+



In [69]:
usage_df.filter(col("data_used_gb").isNull()).show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1015|        105|2026-02-01 00:00:00|        NULL|        1450|      210|
+--------+-----------+-------------------+------------+------------+---------+



In [70]:
payments_df.filter(col("amount_paid").isNull()).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5011|        112|2026-01-01 00:00:00|       NULL|         UPI|       Success|
+----------+-----------+-------------------+-----------+------------+--------------+



In [71]:
payments_df.filter(col("payment_mode").isNull()).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5015|        105|2026-02-01 00:00:00|       1199|        NULL|       Pending|
+----------+-----------+-------------------+-----------+------------+--------------+



In [72]:
usage_df=usage_df.fillna({"data_used_gb":0})

In [73]:
payments_df=payments_df.fillna({"amount_paid":0})

In [74]:
payments_df=payments_df.fillna({"payment_mode":"Not Provided"})

In [75]:
customers_df=customers_df.fillna({"plan_id":"UNKNOWN"})

In [76]:
customers_df=customers_df.withColumn("data_quality_status",
                                     when(col("plan_id")=="UNKNOWN","Incomplete")
                                     .otherwise("Complete"))

In [77]:
usage_df=usage_df.withColumn("data_quality_status",
                             when(col("data_used_gb")==0,"Incomplete")
                             .otherwise("Complete"))

In [78]:
payments_df=payments_df.withColumn("data_quality_status",
                                   when((col("amount_paid")==0)|(col("payment_mode")=="Not Provided"),"Incomplete")
                                   .otherwise("Complete"))

In [79]:
customers_df.write.mode("overwrite").parquet("silver/customers")
usage_df.write.mode("overwrite").parquet("silver/usage")
payments_df.write.mode("overwrite").parquet("silver/payments")

In [80]:
# PART 3
plans_flat=plans_df.select("plan_id","plan_name","monthly_fee","data_limit_gb",col("features.unlimited_calls"),col("features.ott_included"),col("features.roaming"))

In [81]:
plans_flat.select("unlimited_calls").show()

+---------------+
|unlimited_calls|
+---------------+
|           true|
|           true|
|          false|
|           true|
+---------------+



In [82]:
plans_flat.select("ott_included").show()

+------------+
|ott_included|
+------------+
|       false|
|        true|
|       false|
|        true|
+------------+



In [83]:
plans_flat.select("roaming").show()

+-------------+
|      roaming|
+-------------+
|     National|
|     National|
|         NULL|
|International|
+-------------+



In [84]:
plans_flat=plans_flat.fillna({"roaming":"Not Available"})

In [85]:
plans_flat.write.mode("overwrite").parquet("silver/plans")

In [86]:
# PART 4
customers_plans=customers_df.join(plans_flat,"plan_id","left")

In [87]:
customers_usage=customers_df.join(usage_df,"customer_id","left")

In [88]:
customers_payments=customers_df.join(payments_df,"customer_id","left")

In [89]:
customers_df = customers_df.withColumnRenamed(
    "data_quality_status",
    "customer_quality_status"
)
usage_df = usage_df.withColumnRenamed(
    "data_quality_status",
    "usage_quality_status"
)
payments_df = payments_df.withColumnRenamed(
    "data_quality_status",
    "payment_quality_status"
)

In [90]:
final_df=customers_df\
.join(plans_flat,"plan_id","left")\
.join(usage_df,"customer_id","left")\
.join(payments_df,["customer_id"],"left")

In [91]:
customers_df.join(plans_flat,"plan_id","left_anti").show()

+-------+-----------+-------------+---------+-----------+---+------+------+-----------------------+
|plan_id|customer_id|customer_name|     city|      state|age|gender|status|customer_quality_status|
+-------+-----------+-------------+---------+-----------+---+------+------+-----------------------+
|   P105|        111|   Ravi Kumar|   Mumbai|Maharashtra| 45|  Male|Active|               Complete|
|UNKNOWN|        112|  Ayesha Khan|Hyderabad|  Telangana| 28|Female|Active|             Incomplete|
+-------+-----------+-------------+---------+-----------+---+------+------+-----------------------+



In [92]:
usage_df.join(customers_df,"customer_id","left_anti").show()

+-----------+--------+-------------------+------------+------------+---------+--------------------+
|customer_id|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|usage_quality_status|
+-----------+--------+-------------------+------------+------------+---------+--------------------+
|        120|    1011|2026-01-01 00:00:00|          60|        1300|      140|            Complete|
+-----------+--------+-------------------+------------+------------+---------+--------------------+



In [93]:
payments_df.join(customers_df,"customer_id","left_anti").show()

+-----------+----------+----------+-----------+------------+--------------+----------------------+
|customer_id|payment_id|bill_month|amount_paid|payment_mode|payment_status|payment_quality_status|
+-----------+----------+----------+-----------+------------+--------------+----------------------+
+-----------+----------+----------+-----------+------------+--------------+----------------------+



In [94]:
# PART 5
final_df=final_df.withColumn("usage_category",when(col("data_used_gb")>=70,"Heavy User").when(col("data_used_gb")>=30,"Medium User").otherwise("Low User"))

In [95]:
final_df=final_df.withColumn("payment_category",when(col("amount_paid")>=1000,"High Payment").when(col("amount_paid")>=500,"Medium Payment").otherwise("Low Payment"))

In [96]:
final_df=final_df.withColumn("churn_risk",when((col("status")=="Inactive")|(col("payment_status")!="Success"),"High Risk").when(col("data_used_gb")<15,"Medium Risk").otherwise("Low Risk"))

In [97]:
final_df=final_df.withColumn("over_usage_gb",col("data_used_gb")-col("data_limit_gb"))

In [98]:
final_df=final_df.withColumn("over_usage_flag",when(col("over_usage_gb")>0,"Yes").otherwise("No"))

In [99]:
# PART 6
final_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    5|
|    Kochi|    1|
|  Chennai|    4|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    5|
|Hyderabad|    6|
+---------+-----+



In [100]:
final_df.groupBy("state").count().show()

+-----------+-----+
|      state|count|
+-----------+-----+
|  Karnataka|    5|
|     Kerala|    1|
| Tamil Nadu|    4|
|      Delhi|    5|
|  Telangana|    6|
|Maharashtra|    3|
+-----------+-----+



In [101]:
final_df.groupBy("plan_name").count().show()

+------------+-----+
|   plan_name|count|
+------------+-----+
|        NULL|    2|
| Smart Basic|    9|
|Budget Saver|    2|
| Premium Max|    5|
|  Smart Plus|    6|
+------------+-----+



In [102]:
final_df.groupBy("usage_category").count().show()

+--------------+-----+
|usage_category|count|
+--------------+-----+
|   Medium User|   14|
|    Heavy User|    3|
|      Low User|    7|
+--------------+-----+



In [103]:
final_df.groupBy("churn_risk").count().show()

+-----------+-----+
| churn_risk|count|
+-----------+-----+
|   Low Risk|   19|
|Medium Risk|    1|
|  High Risk|    4|
+-----------+-----+



In [104]:
final_df.groupBy("plan_name").agg(sum("data_used_gb")).show()

+------------+-----------------+
|   plan_name|sum(data_used_gb)|
+------------+-----------------+
|        NULL|             NULL|
| Smart Basic|              464|
|Budget Saver|               22|
| Premium Max|              230|
|  Smart Plus|              188|
+------------+-----------------+



In [105]:
final_df.groupBy("plan_name").agg(avg("data_used_gb")).show()

+------------+------------------+
|   plan_name| avg(data_used_gb)|
+------------+------------------+
|        NULL|              NULL|
| Smart Basic| 51.55555555555556|
|Budget Saver|              11.0|
| Premium Max|              46.0|
|  Smart Plus|31.333333333333332|
+------------+------------------+



In [106]:
final_df.groupBy("city").agg(sum("call_minutes")).show()

+---------+-----------------+
|     city|sum(call_minutes)|
+---------+-----------------+
|Bangalore|             3450|
|    Kochi|             1600|
|  Chennai|             4600|
|   Mumbai|              250|
|     Pune|              500|
|    Delhi|             6600|
|Hyderabad|             4000|
+---------+-----------------+



In [107]:
final_df.groupBy("state").agg(sum("sms_count")).show()

+-----------+--------------+
|      state|sum(sms_count)|
+-----------+--------------+
|  Karnataka|           430|
|     Kerala|           250|
| Tamil Nadu|           620|
|      Delhi|           910|
|  Telangana|           520|
|Maharashtra|           100|
+-----------+--------------+



In [108]:
final_df.filter(col("payment_status")=="Success").agg(sum("amount_paid")).show()

+----------------+
|sum(amount_paid)|
+----------------+
|           12882|
+----------------+



In [109]:
final_df.groupBy("city").agg(sum("amount_paid")).show()

+---------+----------------+
|     city|sum(amount_paid)|
+---------+----------------+
|Bangalore|            3695|
|    Kochi|            1199|
|  Chennai|            1996|
|   Mumbai|             299|
|     Pune|             799|
|    Delhi|            5595|
|Hyderabad|            2295|
+---------+----------------+



In [110]:
final_df.groupBy("plan_name").agg(sum("amount_paid")).show()

+------------+----------------+
|   plan_name|sum(amount_paid)|
+------------+----------------+
|        NULL|               0|
| Smart Basic|            4491|
|Budget Saver|             598|
| Premium Max|            5995|
|  Smart Plus|            4794|
+------------+----------------+



In [111]:
final_df.groupBy("payment_mode").agg(sum("amount_paid")).show()

+------------+----------------+
|payment_mode|sum(amount_paid)|
+------------+----------------+
|        NULL|            NULL|
|        Card|            6193|
|        Cash|             598|
|Not Provided|            2398|
|         UPI|            6689|
+------------+----------------+



In [112]:
final_df.groupBy("plan_name").agg(sum("amount_paid").alias("revenue")).orderBy(desc("revenue")).show(1)

+-----------+-------+
|  plan_name|revenue|
+-----------+-------+
|Premium Max|   5995|
+-----------+-------+
only showing top 1 row


In [113]:
final_df.groupBy("city").agg(sum("amount_paid").alias("revenue")).orderBy(desc("revenue")).show(1)

+-----+-------+
| city|revenue|
+-----+-------+
|Delhi|   5595|
+-----+-------+
only showing top 1 row


In [114]:
# PART 7
from pyspark.sql.window import Window
w=Window.orderBy(desc("data_used_gb"))
final_df.withColumn("rank",rank().over(w)).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-----------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+--------------------+----------+-------------------+-----------+------------+--------------+----------------------+--------------+----------------+----------+-------------+---------------+----+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|customer_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|usage_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|payment_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|rank|
+-----------+-------+-------------+---------+-----------+---+------+--------+-----------------------+------------+--

In [115]:
w=Window.orderBy(desc("amount_paid"))
final_df.withColumn("rank",rank().over(w)).show()

+-----------+-------+-------------+---------+-----------+---+------+------+-----------------------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+--------------------+----------+-------------------+-----------+------------+--------------+----------------------+--------------+----------------+-----------+-------------+---------------+----+
|customer_id|plan_id|customer_name|     city|      state|age|gender|status|customer_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|usage_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|payment_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|rank|
+-----------+-------+-------------+---------+-----------+---+------+------+-----------------------+-----------+---------

In [116]:
final_df.orderBy(desc("data_used_gb")).show(3)

+-----------+-------+-------------+-----+------+---+------+------+-----------------------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+--------------------+----------+-------------------+-----------+------------+--------------+----------------------+--------------+----------------+----------+-------------+---------------+
|customer_id|plan_id|customer_name| city| state|age|gender|status|customer_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|usage_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|payment_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|
+-----------+-------+-------------+-----+------+---+------+------+-----------------------+-----------+-----------+-------------+---------------+------

In [117]:
final_df.orderBy(desc("amount_paid")).show(3)

+-----------+-------+-------------+-----+-----+---+------+------+-----------------------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+--------------------+----------+-------------------+-----------+------------+--------------+----------------------+--------------+----------------+-----------+-------------+---------------+
|customer_id|plan_id|customer_name| city|state|age|gender|status|customer_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|usage_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|payment_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|
+-----------+-------+-------------+-----+-----+---+------+------+-----------------------+-----------+-----------+-------------+---------------+-------

In [118]:
w=Window.partitionBy("city").orderBy(desc("data_used_gb"))
final_df.withColumn("rank",rank().over(w)).filter(col("rank")==1).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-----------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+--------------------+----------+-------------------+-----------+------------+--------------+----------------------+--------------+----------------+----------+-------------+---------------+----+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|customer_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|usage_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|payment_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|rank|
+-----------+-------+-------------+---------+-----------+---+------+--------+-----------------------+------------+--

In [119]:
w=Window.partitionBy("plan_name").orderBy(desc("data_used_gb"))
final_df.withColumn("rank",rank().over(w)).filter(col("rank")==1).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-----------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+--------------------+----------+-------------------+-----------+------------+--------------+----------------------+--------------+----------------+----------+-------------+---------------+----+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|customer_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|usage_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|payment_quality_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|rank|
+-----------+-------+-------------+---------+-----------+---+------+--------+-----------------------+------------+--

In [120]:
monthly_rev=payments_df.groupBy("bill_month").agg(sum("amount_paid").alias("revenue"))
monthly_rev.withColumn("running_total",sum("revenue").over(Window.orderBy("bill_month"))).show()

+-------------------+-------+-------------+
|         bill_month|revenue|running_total|
+-------------------+-------+-------------+
|2026-01-01 00:00:00|   6890|         6890|
|2026-02-01 00:00:00|   2996|         9886|
+-------------------+-------+-------------+



In [121]:
w=Window.partitionBy("customer_id").orderBy("usage_month")
usage_df.withColumn("previous_usage",lag("data_used_gb").over(w)).show()

+--------+-----------+-------------------+------------+------------+---------+--------------------+--------------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|usage_quality_status|previous_usage|
+--------+-----------+-------------------+------------+------------+---------+--------------------+--------------+
|    1001|        101|2026-01-01 00:00:00|          45|         900|      120|            Complete|          NULL|
|    1012|        101|2026-02-01 00:00:00|          50|        1000|      130|            Complete|            45|
|    1002|        102|2026-01-01 00:00:00|          30|         600|       80|            Complete|          NULL|
|    1013|        102|2026-02-01 00:00:00|          34|         650|       85|            Complete|            30|
|    1003|        103|2026-01-01 00:00:00|          12|         250|       40|            Complete|          NULL|
|    1004|        104|2026-01-01 00:00:00|          55|        1100|      150|  

In [122]:
usage_df.withColumn("next_usage",lead("data_used_gb").over(w)).show()

+--------+-----------+-------------------+------------+------------+---------+--------------------+----------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|usage_quality_status|next_usage|
+--------+-----------+-------------------+------------+------------+---------+--------------------+----------+
|    1001|        101|2026-01-01 00:00:00|          45|         900|      120|            Complete|        50|
|    1012|        101|2026-02-01 00:00:00|          50|        1000|      130|            Complete|      NULL|
|    1002|        102|2026-01-01 00:00:00|          30|         600|       80|            Complete|        34|
|    1013|        102|2026-02-01 00:00:00|          34|         650|       85|            Complete|      NULL|
|    1003|        103|2026-01-01 00:00:00|          12|         250|       40|            Complete|      NULL|
|    1004|        104|2026-01-01 00:00:00|          55|        1100|      150|            Complete|        58|
|

In [123]:
usage_df.withColumn("previous_usage",lag("data_used_gb").over(w)).filter(col("data_used_gb")>col("previous_usage")).show()

+--------+-----------+-------------------+------------+------------+---------+--------------------+--------------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|usage_quality_status|previous_usage|
+--------+-----------+-------------------+------------+------------+---------+--------------------+--------------+
|    1012|        101|2026-02-01 00:00:00|          50|        1000|      130|            Complete|            45|
|    1013|        102|2026-02-01 00:00:00|          34|         650|       85|            Complete|            30|
|    1014|        104|2026-02-01 00:00:00|          58|        1200|      160|            Complete|            55|
+--------+-----------+-------------------+------------+------------+---------+--------------------+--------------+



In [124]:
# PART 8
customers_df.createOrReplaceTempView("customers")
usage_df.createOrReplaceTempView("usage")
payments_df.createOrReplaceTempView("payments")
plans_flat.createOrReplaceTempView("plans")

In [125]:
spark.sql("select * from customers where status='Active'").show()

+-----------+-------------+---------+-----------+---+------+-------+------+-----------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|status|customer_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+------+-----------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|Active|               Complete|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|Active|               Complete|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|Active|               Complete|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|Active|               Complete|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|Active|               Complete|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|Active|               Complete|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|Active|               Complete|


In [126]:
spark.sql("select city,count(*) total_customers from customers group by city").show()

+---------+---------------+
|     city|total_customers|
+---------+---------------+
|Bangalore|              2|
|    Kochi|              1|
|  Chennai|              1|
|   Mumbai|              2|
|     Pune|              1|
|    Delhi|              2|
|Hyderabad|              3|
+---------+---------------+



In [127]:
spark.sql("""
select p.plan_name,sum(py.amount_paid) revenue
from customers c
join plans p on c.plan_id=p.plan_id
join payments py on c.customer_id=py.customer_id
group by p.plan_name
""").show()

+------------+-------+
|   plan_name|revenue|
+------------+-------+
| Smart Basic|   2495|
|Budget Saver|    598|
| Premium Max|   3597|
|  Smart Plus|   3196|
+------------+-------+



In [128]:
spark.sql("select * from usage where data_used_gb>=70").show()

+--------+-----------+-------------------+------------+------------+---------+--------------------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|usage_quality_status|
+--------+-----------+-------------------+------------+------------+---------+--------------------+
|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|            Complete|
|    1008|        108|2026-01-01 00:00:00|          80|        1600|      250|            Complete|
+--------+-----------+-------------------+------------+------------+---------+--------------------+



In [129]:
spark.sql("""
select *
from customers c
left join payments p
on c.customer_id=p.customer_id
where c.status='Inactive' or p.payment_status<>'Success'
""").show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-----------------------+----------+-----------+-------------------+-----------+------------+--------------+----------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|customer_quality_status|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|payment_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-----------------------+----------+-----------+-------------------+-----------+------------+--------------+----------------------+
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|               Complete|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|              Complete|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|               Complete|      5015|        105|2026-02-01 00:00:00|       1199|Not Provided

In [130]:
spark.sql("""
select *
from customers
where plan_id='UNKNOWN'
""").show()

+-----------+-------------+---------+---------+---+------+-------+------+-----------------------+
|customer_id|customer_name|     city|    state|age|gender|plan_id|status|customer_quality_status|
+-----------+-------------+---------+---------+---+------+-------+------+-----------------------+
|        112|  Ayesha Khan|Hyderabad|Telangana| 28|Female|UNKNOWN|Active|             Incomplete|
+-----------+-------------+---------+---------+---+------+-------+------+-----------------------+



In [131]:
spark.sql("""
select *
from payments
where payment_status in ('Failed','Pending')
""").show()

+----------+-----------+-------------------+-----------+------------+--------------+----------------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|payment_quality_status|
+----------+-----------+-------------------+-----------+------------+--------------+----------------------+
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|              Complete|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|              Complete|
|      5015|        105|2026-02-01 00:00:00|       1199|Not Provided|       Pending|            Incomplete|
+----------+-----------+-------------------+-----------+------------+--------------+----------------------+



In [132]:
spark.sql("""
select customer_id,sum(data_used_gb) total_usage
from usage
group by customer_id
order by total_usage desc
limit 5
""").show()

+-----------+-----------+
|customer_id|total_usage|
+-----------+-----------+
|        104|        113|
|        101|         95|
|        108|         80|
|        105|         75|
|        102|         64|
+-----------+-----------+



In [133]:
spark.sql("""
select payment_mode,sum(amount_paid)
from payments
group by payment_mode
""").show()

+------------+----------------+
|payment_mode|sum(amount_paid)|
+------------+----------------+
|        Card|            3696|
|        Cash|             598|
|Not Provided|            1199|
|         UPI|            4393|
+------------+----------------+



In [134]:
# PART 9
final_df.write.mode("overwrite").parquet("gold/customer_analytics")

In [135]:
final_df.write.mode("overwrite").partitionBy("usage_month").parquet("gold/customer_analytics_partitioned")

In [136]:
incremental_df.show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1016|        101|2026-03-01 00:00:00|          52|        1050|      135|
|    1017|        102|2026-03-01 00:00:00|          38|         700|       95|
|    1018|        104|2026-03-01 00:00:00|          62|        1250|      170|
|    1019|        105|2026-03-01 00:00:00|          78|        1550|      220|
|    1020|        108|2026-03-01 00:00:00|          85|        1700|      260|
+--------+-----------+-------------------+------------+------------+---------+



In [137]:
incremental_df.write.mode("append").parquet("silver/usage")

In [138]:
silver_usage=spark.read.parquet("silver/usage")

In [139]:
silver_usage = silver_usage \
    .withColumn("data_used_gb", col("data_used_gb").cast("double")) \
    .withColumn("call_minutes", col("call_minutes").cast("int")) \
    .withColumn("sms_count", col("sms_count").cast("int"))

In [140]:
silver_usage.groupBy("customer_id").agg(sum("data_used_gb")).show()

+-----------+-----------------+
|customer_id|sum(data_used_gb)|
+-----------+-----------------+
|        108|            165.0|
|        101|            147.0|
|        103|             12.0|
|        120|             60.0|
|        107|             10.0|
|        102|            102.0|
|        109|             48.0|
|        105|            153.0|
|        110|             32.0|
|        106|             28.0|
|        104|            175.0|
+-----------+-----------------+



In [141]:
final_df.write.mode("overwrite").partitionBy("usage_month").parquet("gold/customer_analytics_partitioned")

In [142]:
print(usage_df.count(),silver_usage.count())

15 20


In [145]:
# PART 10
customer_usage_summary=final_df.select("customer_id","customer_name","city","plan_name","usage_month","data_used_gb","data_limit_gb","over_usage_flag","amount_paid","payment_status","churn_risk")
customer_usage_summary.show()

+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+-----------+
|customer_id|customer_name|     city|   plan_name|        usage_month|data_used_gb|data_limit_gb|over_usage_flag|amount_paid|payment_status| churn_risk|
+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+-----------+
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-02-01 00:00:00|          50|           50|             No|        499|       Success|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-02-01 00:00:00|          50|           50|             No|        499|       Success|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-01-01 00:00:00|          45|           50|             No|        499|       Success|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-01-01 00:00:00|          45

In [154]:
from pyspark.sql.functions import countDistinct, sum, avg
plan_performance=final_df.groupBy("plan_name").agg(countDistinct("customer_id"),sum("data_used_gb"),avg("data_used_gb"),sum("amount_paid"))
plan_performance.show()

+------------+---------------------------+-----------------+------------------+----------------+
|   plan_name|count(DISTINCT customer_id)|sum(data_used_gb)| avg(data_used_gb)|sum(amount_paid)|
+------------+---------------------------+-----------------+------------------+----------------+
|        NULL|                          2|             NULL|              NULL|               0|
| Smart Basic|                          3|              464| 51.55555555555556|            4491|
|Budget Saver|                          2|               22|              11.0|             598|
| Premium Max|                          2|              230|              46.0|            5995|
|  Smart Plus|                          3|              188|31.333333333333332|            4794|
+------------+---------------------------+-----------------+------------------+----------------+



In [155]:
city_revenue=final_df.groupBy("city").agg(countDistinct("customer_id"),sum("amount_paid"),avg("amount_paid"))
city_revenue.show()

+---------+---------------------------+----------------+----------------+
|     city|count(DISTINCT customer_id)|sum(amount_paid)|avg(amount_paid)|
+---------+---------------------------+----------------+----------------+
|Bangalore|                          2|            3695|           739.0|
|    Kochi|                          1|            1199|          1199.0|
|  Chennai|                          1|            1996|           499.0|
|   Mumbai|                          2|             299|           299.0|
|     Pune|                          1|             799|           799.0|
|    Delhi|                          2|            5595|          1119.0|
|Hyderabad|                          3|            2295|           382.5|
+---------+---------------------------+----------------+----------------+



In [152]:
churn_report=final_df.select("customer_id","customer_name","city","plan_name","payment_status","status","churn_risk")
churn_report.show()

+-----------+-------------+---------+------------+--------------+--------+-----------+
|customer_id|customer_name|     city|   plan_name|payment_status|  status| churn_risk|
+-----------+-------------+---------+------------+--------------+--------+-----------+
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|   Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|   Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|   Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|   Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|   Low Risk|
|        103|   Amit Kumar|   Mumbai|Budget

In [153]:
over_usage_report=final_df.select("customer_id","customer_name","plan_name","data_used_gb","data_limit_gb","over_usage_gb")
over_usage_report.show()

+-----------+-------------+------------+------------+-------------+-------------+
|customer_id|customer_name|   plan_name|data_used_gb|data_limit_gb|over_usage_gb|
+-----------+-------------+------------+------------+-------------+-------------+
|        101| Rahul Sharma| Smart Basic|          50|           50|            0|
|        101| Rahul Sharma| Smart Basic|          50|           50|            0|
|        101| Rahul Sharma| Smart Basic|          45|           50|           -5|
|        101| Rahul Sharma| Smart Basic|          45|           50|           -5|
|        102|  Priya Reddy|  Smart Plus|          34|           75|          -41|
|        102|  Priya Reddy|  Smart Plus|          34|           75|          -41|
|        102|  Priya Reddy|  Smart Plus|          30|           75|          -45|
|        102|  Priya Reddy|  Smart Plus|          30|           75|          -45|
|        103|   Amit Kumar|Budget Saver|          12|           25|          -13|
|        104|  S